In [2]:
# Feasibility Analysis for RQ3 - inspection of context responses
import pandas as pd
from pathlib import Path
df_feas_ctx = pd.read_csv('../2_context/contextResponses/contextFeasibilityResponses.csv')

df_feas_ctx.info()

unique_ctx = df_feas_ctx['iteration'].nunique()
print(f"Unique iteration in context responses: {unique_ctx}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   row_id        600 non-null    object
 1   variant_id    600 non-null    object
 2   base_model    600 non-null    object
 3   model         600 non-null    object
 4   rating        600 non-null    int64 
 5   label         600 non-null    object
 6   iteration     600 non-null    int64 
 7   timestamp     600 non-null    object
 8   raw_response  600 non-null    object
dtypes: int64(2), object(7)
memory usage: 42.3+ KB
Unique iteration in context responses: 50


In [3]:
# Feasibility Analysis for RQ3 - inspection of anchor responses
import pandas as pd
from pathlib import Path
df_feas_anchor = pd.read_csv('../3_anchor/anchorResponses/anchorFeasibilityResponses.csv')

df_feas_anchor.info()

# count unique iterations per arm there are 4 arms in the anchor responses
print(f"Unique iterations: {df_feas_anchor['iteration'].nunique()}")
for arm in df_feas_anchor['anchor_type'].unique():
    print(f"Unique iterations for arm {arm}: {df_feas_anchor[df_feas_anchor['anchor_type'] == arm]['iteration'].nunique()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   row_id        600 non-null    object
 1   base_model    600 non-null    object
 2   variant_id    600 non-null    object
 3   model         600 non-null    object
 4   rating        600 non-null    int64 
 5   label         600 non-null    object
 6   iteration     600 non-null    int64 
 7   timestamp     600 non-null    object
 8   raw_response  600 non-null    object
 9   condition     600 non-null    object
 10  anchor_type   600 non-null    object
 11  anchor_level  360 non-null    object
dtypes: int64(2), object(10)
memory usage: 56.4+ KB
Unique iterations: 20
Unique iterations for arm ANCHOR_WORD: 10
Unique iterations for arm ANCHOR_EXAMPLE: 10
Unique iterations for arm ANCHOR_NUM_LOW: 10
Unique iterations for arm ANCHOR_NUM_HIGH: 20


In [4]:
from pathlib import Path
import pandas as pd

# Create output directory
output_dir = Path('feas-ctx-anchor-working')
output_dir.mkdir(parents=True, exist_ok=True)

# Ensure iteration is numeric
df_feas_anchor['iteration'] = pd.to_numeric(df_feas_anchor['iteration'], errors='coerce')

# Keep all rows from the other arms.
df_feas_anchor = df_feas_anchor[(df_feas_anchor['anchor_type'] != 'ANCHOR_NUM_HIGH') | ((df_feas_anchor['anchor_type'] == 'ANCHOR_NUM_HIGH') & (df_feas_anchor['iteration'] < 10))].copy()

# Save trimmed data
output_path = output_dir / 'anchorFeasibilityResponses_10each.csv'
df_feas_anchor.to_csv(output_path, index=False)

# Audit iterations and rows per arm
# count unique iterations per arm there are 4 arms in the anchor responses
print(f"Unique iterations: {df_feas_anchor['iteration'].nunique()}")
for arm in df_feas_anchor['anchor_type'].unique():
    print(f"Unique iterations for arm {arm}: {df_feas_anchor[df_feas_anchor['anchor_type'] == arm]['iteration'].nunique()}")

Unique iterations: 10
Unique iterations for arm ANCHOR_WORD: 10
Unique iterations for arm ANCHOR_EXAMPLE: 10
Unique iterations for arm ANCHOR_NUM_LOW: 10
Unique iterations for arm ANCHOR_NUM_HIGH: 10


In [5]:
df_feas_anchor = pd.read_csv('feas-ctx-anchor-working/anchorFeasibilityResponses_10each.csv')
df_feas_anchor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   row_id        480 non-null    object
 1   base_model    480 non-null    object
 2   variant_id    480 non-null    object
 3   model         480 non-null    object
 4   rating        480 non-null    int64 
 5   label         480 non-null    object
 6   iteration     480 non-null    int64 
 7   timestamp     480 non-null    object
 8   raw_response  480 non-null    object
 9   condition     480 non-null    object
 10  anchor_type   480 non-null    object
 11  anchor_level  240 non-null    object
dtypes: int64(2), object(10)
memory usage: 45.1+ KB


In [7]:
# cell3: merging anchor and context data of feasibility responses
import pandas as pd

# load data
df_feas_ctx = pd.read_csv('../2_context/contextResponses/contextFeasibilityResponses.csv')
df_feas_anchor = pd.read_csv('feas-ctx-anchor-working/anchorFeasibilityResponses_10each.csv')

# define source and condition
df_feas_ctx['condition'] = 'context'
df_feas_anchor['condition'] = 'anchor'

# exact columns to keep
columns_to_keep = ['row_id', 'variant_id', 'base_model', 'model', 'rating', 'label', 'condition', 'anchor_type', 'anchor_level', 'iteration']

# add any missing columns
for column in columns_to_keep:
    if column not in df_feas_ctx.columns: df_feas_ctx[column] = pd.NA
    if column not in df_feas_anchor.columns: df_feas_anchor[column] = pd.NA

# combine the dataframes
df_combined = pd.concat([df_feas_ctx[columns_to_keep], df_feas_anchor[columns_to_keep]], ignore_index=True)

# save combined dataframe
df_combined.to_csv(output_dir / 'feas-ctx-anchor-responses.csv', index=False)

df_combined.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1080 entries, 0 to 1079
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   row_id        1080 non-null   object
 1   variant_id    1080 non-null   object
 2   base_model    1080 non-null   object
 3   model         1080 non-null   object
 4   rating        1080 non-null   int64 
 5   label         1080 non-null   object
 6   condition     1080 non-null   object
 7   anchor_type   480 non-null    object
 8   anchor_level  240 non-null    object
 9   iteration     1080 non-null   int64 
dtypes: int64(2), object(8)
memory usage: 84.5+ KB


In [1]:
# Bayesian analysis of RQ3 feasibility: primary results table   
from pathlib import Path
import pandas as pd
from IPython.display import display

# Paths are relative to rq3-rerun/rq3-feas-analysis.ipynb
working_dir = Path("feas-ctx-anchor-working")

input_path = (
    working_dir
    / "bayesian-results"
    / "prior-sensitivity"
    / "rq3_feasibility_prior_sensitivity_effects.csv"
)

output_path = (
    working_dir
    / "rq3_feasibility_primary_results_table.tex"
)

if not input_path.exists():
    raise FileNotFoundError(
        f"RQ3 sensitivity results not found:\n{input_path.resolve()}"
    )

model_labels = {
    "gemma3": "Gemma3:12B",
    "llama": "LLaMa-Pro",
    "mistral": "Mistral",
    "phi4": "Phi-4",
}

model_order = {
    "phi4": 1,
    "gemma3": 2,
    "mistral": 3,
    "llama": 4,
}

anchor_labels = {
    "ANCHOR_EXAMPLE": "Example",
    "ANCHOR_WORD": "Word",
    "ANCHOR_NUM_HIGH": "Number (high)",
    "ANCHOR_NUM_LOW": "Number (low)",
}

anchor_order = {
    "ANCHOR_EXAMPLE": 1,
    "ANCHOR_WORD": 2,
    "ANCHOR_NUM_HIGH": 3,
    "ANCHOR_NUM_LOW": 4,
}

conclusion_labels = {
    "higher under Anchor": "Higher under anchor",
    "lower under Anchor": "Lower under anchor",
    "uncertain": "Uncertain",
}


def format_probability(value):
    """Avoid displaying a posterior draw proportion as absolute certainty."""
    value = float(value)

    if value >= 0.9995:
        return r"$>0.999$"
    if value <= 0.0005:
        return r"$<0.001$"

    return f"{value:.3f}"


# Read the complete prior-sensitivity results
results = pd.read_csv(input_path)

required_columns = {
    "prior_name",
    "base_model",
    "anchor_type",
    "OR_median",
    "OR_l95",
    "OR_u95",
    "posterior_probability_OR_gt_1",
    "conclusion",
}

missing_columns = required_columns.difference(results.columns)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

# The primary-prior estimates are used for the main results table
primary = results.loc[
    results["prior_name"].eq("primary")
].copy()

if len(primary) != 16:
    raise ValueError(
        f"Expected 16 RQ3 primary rows, found {len(primary)}."
    )

if set(primary["base_model"]) != set(model_labels):
    raise ValueError(
        "The RQ3 table does not contain the expected four base models."
    )

if set(primary["anchor_type"]) != set(anchor_labels):
    raise ValueError(
        "The RQ3 table does not contain the expected four anchor types."
    )

primary["model_order"] = primary["base_model"].map(model_order)
primary["anchor_order"] = primary["anchor_type"].map(anchor_order)

primary = (
    primary
    .sort_values("model_order")
    .reset_index(drop=True)
)

table = pd.DataFrame({
    "Model": primary["base_model"].map(model_labels),
    "Anchor type": primary["anchor_type"].map(anchor_labels),
    "OR": primary["OR_median"].map(
        lambda value: f"{value:.3f}"
    ),
    r"95\% CrI": [
        f"[{lower:.3f}, {upper:.3f}]"
        for lower, upper in zip(
            primary["OR_l95"],
            primary["OR_u95"],
        )
    ],
    r"$P(OR>1)$": primary[
        "posterior_probability_OR_gt_1"
    ].map(format_probability),
    "Interpretation": primary["conclusion"].map(
        conclusion_labels
    ),
})

latex = table.to_latex(
    index=False,
    escape=False,
    column_format="llcccc",
    caption=(
        "RQ3 feasibility: posterior effects of each anchor type "
        "relative to Context under the primary prior."
    ),
    label="tab:rq3_feasibility_bayesian",
    position="tbp",
)

# Use the full width in a two-column manuscript
latex = latex.replace(
    r"\begin{table}",
    r"\begin{table*}",
    1,
).replace(
    r"\end{table}",
    r"\end{table*}",
    1,
)

note = (
    "\\end{tabular}\n"
    "\\vspace{2pt}\n"
    "\\begin{minipage}{\\linewidth}\n"
    "\\footnotesize\\textit{Note.} "
    "OR $>1$ indicates higher odds of receiving a higher "
    "feasibility rating under the named anchor type relative "
    "to Context. CrI denotes the 95\\% Bayesian credible "
    "interval. The primary effect prior was Normal$(0,1.5)$.\n"
    "\\end{minipage}"
)

latex = latex.replace(
    "\\end{tabular}",
    note,
    1,
)

output_path.write_text(latex, encoding="utf-8")

print(f"RQ3 LaTeX table saved to:\n{output_path.resolve()}")
display(table)

RQ3 LaTeX table saved to:
/Users/HP/Documents/second-publication/current-analysis/re-run/rq3-rerun/feas-ctx-anchor-working/rq3_feasibility_primary_results_table.tex


,Model,Anchor type,OR,95\% CrI,$P(OR>1)$,Interpretation
0,Phi-4,Example,1.210,"[0.156, 10.258]",0.571,Uncertain
1,Phi-4,Number (high),1.208,"[0.166, 10.510]",0.570,Uncertain
2,Phi-4,Number (low),1.228,"[0.151, 10.851]",0.573,Uncertain
3,Phi-4,Word,1.218,"[0.160, 10.844]",0.574,Uncertain
4,Gemma3:12B,Example,1.227,"[0.162, 10.743]",0.577,Uncertain
5,Gemma3:12B,Number (high),1.257,"[0.163, 10.513]",0.579,Uncertain
6,Gemma3:12B,Number (low),1.260,"[0.168, 11.360]",0.581,Uncertain
7,Gemma3:12B,Word,1.226,"[0.155, 10.752]",0.574,Uncertain
8,Mistral,Example,1.145,"[0.151, 9.070]",0.555,Uncertain
9,Mistral,Number (high),1.168,"[0.160, 9.160]",0.560,Uncertain


In [2]:
# rq3 feasibility: sensitivity comparisons for the online appendix
from pathlib import Path
import pandas as pd
from IPython.display import display

working_dir = Path("feas-ctx-anchor-working")

input_path = (
    working_dir
    / "bayesian-results"
    / "prior-sensitivity"
    / "rq3_feasibility_prior_sensitivity_effects.csv"
)

output_path = (
    working_dir
    / "rq3_feasibility_prior_sensitivity_table.tex"
)

if not input_path.exists():
    raise FileNotFoundError(
        f"Sensitivity results not found:\n{input_path.resolve()}"
    )

model_labels = {
    "gemma3": "Gemma3:12B",
    "llama": "LLaMa-Pro",
    "mistral": "Mistral",
    "phi4": "Phi-4",
}

model_order = [
    "phi4",
    "gemma3",
    "mistral",
    "llama",
]

anchor_labels = {
    "ANCHOR_EXAMPLE": "Example",
    "ANCHOR_WORD": "Word",
    "ANCHOR_NUM_HIGH": "High numeric",
    "ANCHOR_NUM_LOW": "Low numeric",
}

anchor_order = [
    "ANCHOR_EXAMPLE",
    "ANCHOR_WORD",
    "ANCHOR_NUM_HIGH",
    "ANCHOR_NUM_LOW",
]

prior_order = [
    "regularizing",
    "primary",
    "weak",
]

prior_labels = {
    "regularizing": "Regularizing",
    "primary": "Primary",
    "weak": "Weak",
}

conclusion_labels = {
    "higher under Anchor": "Higher under Anchor",
    "lower under Anchor": "Lower under Anchor",
    "uncertain": "Uncertain",
}


def format_number(value):
    value = float(value)

    if value != 0 and (
        abs(value) < 0.001 or abs(value) >= 1000
    ):
        mantissa, exponent = f"{value:.2e}".split("e")
        return (
            rf"${mantissa}\times 10^{{{int(exponent)}}}$"
        )

    return f"{value:.3f}"


def format_effect(row):
    return (
        f"{format_number(row['OR_median'])} "
        f"[{format_number(row['OR_l95'])}, "
        f"{format_number(row['OR_u95'])}]"
    )


results = pd.read_csv(input_path)

required_columns = {
    "prior_name",
    "base_model",
    "anchor_type",
    "OR_median",
    "OR_l95",
    "OR_u95",
    "conclusion",
}

missing_columns = required_columns.difference(
    results.columns
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

if len(results) != 48:
    raise ValueError(
        "Expected 48 rows: 4 models × 4 anchor types "
        f"× 3 priors; found {len(results)}."
    )

records = []

for base_model in model_order:
    for anchor_type in anchor_order:
        contrast_data = results.loc[
            results["base_model"].eq(base_model)
            & results["anchor_type"].eq(anchor_type)
        ]

        if set(contrast_data["prior_name"]) != set(
            prior_order
        ):
            raise ValueError(
                "Incomplete prior results for "
                f"{base_model}/{anchor_type}."
            )

        record = {
            "Model": model_labels[base_model],
            "Anchor type": anchor_labels[anchor_type],
        }

        conclusions = []

        for prior_name in prior_order:
            prior_row = contrast_data.loc[
                contrast_data["prior_name"].eq(prior_name)
            ].iloc[0]

            record[
                f"{prior_labels[prior_name]} OR (95\\% CrI)"
            ] = format_effect(prior_row)

            conclusions.append(
                conclusion_labels[
                    prior_row["conclusion"]
                ]
            )

        record["Conclusions (R/P/W)"] = " / ".join(
            conclusions
        )

        record["Stable"] = (
            "Yes"
            if len(set(conclusions)) == 1
            else "No"
        )

        records.append(record)

table = pd.DataFrame(records)

latex = table.to_latex(
    index=False,
    escape=False,
    longtable=True,
    column_format=(
        r"p{2.0cm}"
        r"p{2.2cm}"
        r"p{3.1cm}"
        r"p{3.1cm}"
        r"p{3.1cm}"
        r"p{3.7cm}"
        r"c"
    ),
    caption=(
        "Prior-sensitivity comparison for the RQ3 feasibility "
        "effects of each anchor type relative to Context."
    ),
    label="tab:rq3_feasibility_prior_sensitivity",
    position="tbp",
)

note = (
    "\n"
    "\\noindent\\begin{minipage}{\\linewidth}\n"
    "\\footnotesize\\textit{Note.} "
    "Cells report posterior median odds ratios with 95\\% "
    "credible intervals. R/P/W denotes regularizing, primary, "
    "and weak priors, respectively. Stable indicates that the "
    "inferential conclusion was identical under all three priors. "
    "OR $>1$ indicates higher odds of a higher feasibility rating "
    "under the specified anchor relative to Context.\n"
    "\\end{minipage}\n"
)

latex = latex + note

output_path.write_text(latex, encoding="utf-8")

print(f"Saved to:\n{output_path.resolve()}")
display(table)

Saved to:
/Users/HP/Documents/second-publication/current-analysis/re-run/rq3-rerun/feas-ctx-anchor-working/rq3_feasibility_prior_sensitivity_table.tex


,Model,Anchor type,Regularizing OR (95\% CrI),Primary OR (95\% CrI),Weak OR (95\% CrI),Conclusions (R/P/W),Stable
0,Phi-4,Example,"1.091 [0.284, 3.934]","1.210 [0.156, 10.258]","1.450 [0.110, 24.053]",Uncertain / Uncertain / Uncertain,Yes
1,Phi-4,Word,"1.067 [0.292, 4.122]","1.218 [0.160, 10.844]","1.455 [0.120, 21.459]",Uncertain / Uncertain / Uncertain,Yes
2,Phi-4,High numeric,"1.067 [0.292, 4.074]","1.208 [0.166, 10.510]","1.457 [0.115, 21.753]",Uncertain / Uncertain / Uncertain,Yes
3,Phi-4,Low numeric,"1.052 [0.280, 4.040]","1.228 [0.151, 10.851]","1.435 [0.123, 22.042]",Uncertain / Uncertain / Uncertain,Yes
4,Gemma3:12B,Example,"1.077 [0.283, 4.131]","1.227 [0.162, 10.743]","1.443 [0.116, 20.737]",Uncertain / Uncertain / Uncertain,Yes
5,Gemma3:12B,Word,"1.076 [0.287, 4.059]","1.226 [0.155, 10.752]","1.501 [0.115, 21.662]",Uncertain / Uncertain / Uncertain,Yes
6,Gemma3:12B,High numeric,"1.069 [0.287, 4.011]","1.257 [0.163, 10.513]","1.462 [0.114, 20.139]",Uncertain / Uncertain / Uncertain,Yes
7,Gemma3:12B,Low numeric,"1.071 [0.299, 3.887]","1.260 [0.168, 11.360]","1.403 [0.121, 21.725]",Uncertain / Uncertain / Uncertain,Yes
8,Mistral,Example,"1.038 [0.281, 3.766]","1.145 [0.151, 9.070]","1.324 [0.114, 17.059]",Uncertain / Uncertain / Uncertain,Yes
9,Mistral,Word,"1.574 [0.441, 5.722]","3.084 [0.404, 21.888]","5.407 [0.414, 56.770]",Uncertain / Uncertain / Uncertain,Yes


### This produces four Anchor-versus-Context contrasts for each base model.

In [1]:
# anchor effects
from pathlib import Path
import pandas as pd
from IPython.display import display

# Paths are relative to rq3-feas-analysis.ipynb
working_dir = Path("feas-ctx-anchor-working")
results_dir = (
    working_dir
    / "bayesian-results"
    / "anchor-response-patterns"
)
output_dir = results_dir / "latex-tables"
output_dir.mkdir(parents=True, exist_ok=True)

extremity_path = (
    results_dir
    / "rq3_feasibility_primary_extremity_contrasts.csv"
)
numeric_path = (
    results_dir
    / "rq3_feasibility_primary_numeric_anchoring_patterns.csv"
)
extremity_robustness_path = (
    results_dir
    / "rq3_feasibility_extremity_robustness.csv"
)
numeric_robustness_path = (
    results_dir
    / "rq3_feasibility_numeric_anchoring_robustness.csv"
)

required_paths = [
    extremity_path,
    numeric_path,
    extremity_robustness_path,
    numeric_robustness_path,
]

missing_paths = [p for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing RQ3 feasibility results:\n"
        + "\n".join(str(p.resolve()) for p in missing_paths)
    )

model_labels = {
    "gemma3": "Gemma3:12B",
    "llama": "LLaMa-Pro",
    "mistral": "Mistral",
    "phi4": "Phi-4",
}
model_order = ["gemma3", "llama", "mistral", "phi4"]

anchor_labels = {
    "ANCHOR_EXAMPLE": "Example",
    "ANCHOR_WORD": "Word",
}

pattern_labels = {
    "complete directional anchoring": "Complete directional anchoring",
    "high-arm-only adjustment": "High-arm-only adjustment",
    "low-arm-only adjustment": "Low-arm-only adjustment",
    "no supported adjustment": "No supported adjustment",
    "directionally inconsistent": "Directionally inconsistent",
}

extremity_labels = {
    "greater endpoint concentration under Anchor":
        "Greater endpoint concentration",
    "reduced endpoint concentration under Anchor":
        "Reduced endpoint concentration",
    "uncertain": "Uncertain",
}


def format_probability(value):
    value = float(value)
    if value >= 0.9995:
        return r"$>0.999$"
    if value <= 0.0005:
        return r"$<0.001$"
    return f"{value:.3f}"


def format_delta_interval(row):
    return (
        f"{row['difference_median']:+.3f} "
        f"[{row['difference_l95']:.3f}, "
        f"{row['difference_u95']:.3f}]"
    )


def format_or_interval(row, prefix):
    return (
        f"{row[f'{prefix}_OR_median']:.3f} "
        f"[{row[f'{prefix}_OR_l95']:.3f}, "
        f"{row[f'{prefix}_OR_u95']:.3f}]"
    )


def save_regular_table(
    table,
    path,
    caption,
    label,
    column_format,
    note,
    wide=False,
):
    latex = table.to_latex(
        index=False,
        escape=False,
        column_format=column_format,
        caption=caption,
        label=label,
        position="tbp",
    )

    note_block = (
        "\n\\vspace{2pt}\n"
        "\\begin{minipage}{\\linewidth}\n"
        "\\footnotesize\\textit{Note.} "
        + note
        + "\n\\end{minipage}"
    )

    latex = latex.replace(
        "\\end{tabular}",
        "\\end{tabular}" + note_block,
        1,
    )

    if wide:
        latex = latex.replace(
            "\\begin{table}",
            "\\begin{table*}",
            1,
        )
        latex = latex.replace(
            "\\end{table}",
            "\\end{table*}",
            1,
        )

    path.write_text(latex, encoding="utf-8")


# ------------------------------------------------------------------
# Read and validate results
# ------------------------------------------------------------------

extremity = pd.read_csv(extremity_path)
numeric = pd.read_csv(numeric_path)
extremity_robustness = pd.read_csv(extremity_robustness_path)
numeric_robustness = pd.read_csv(numeric_robustness_path)

if len(extremity) != 8:
    raise ValueError(
        f"Expected 8 primary feasibility extremity rows; "
        f"found {len(extremity)}."
    )

if len(numeric) != 4:
    raise ValueError(
        f"Expected 4 primary feasibility numeric rows; "
        f"found {len(numeric)}."
    )

if set(extremity["base_model"]) != set(model_order):
    raise ValueError("Unexpected feasibility base-model values.")

# ------------------------------------------------------------------
# Main-paper compact table
# ------------------------------------------------------------------

extremity = extremity.copy()
extremity["result_cell"] = extremity.apply(
    lambda row: (
        f"{format_delta_interval(row)}; "
        f"{extremity_labels[row['conclusion']]}"
    ),
    axis=1,
)

extremity_wide = extremity.pivot(
    index="base_model",
    columns="anchor_type",
    values="result_cell",
)

numeric_lookup = numeric.set_index("base_model")

main_table = pd.DataFrame({
    "Model": [model_labels[m] for m in model_order],
    r"Example $\Delta_{\mathrm{endpoint}}$ [95\% CrI]": [
        extremity_wide.loc[m, "ANCHOR_EXAMPLE"]
        for m in model_order
    ],
    r"Word $\Delta_{\mathrm{endpoint}}$ [95\% CrI]": [
        extremity_wide.loc[m, "ANCHOR_WORD"]
        for m in model_order
    ],
    "Numeric anchoring pattern": [
        pattern_labels[
            numeric_lookup.loc[m, "anchoring_pattern"]
        ]
        for m in model_order
    ],
})

main_path = (
    output_dir
    / "rq3_feasibility_anchor_patterns_main.tex"
)

save_regular_table(
    table=main_table,
    path=main_path,
    caption=(
        "RQ3 feasibility: anchor-associated endpoint changes "
        "and numeric anchoring patterns."
    ),
    label="tab:rq3_feasibility_anchor_patterns",
    column_format="lccc",
    note=(
        "$\\Delta_{\\mathrm{endpoint}}$ is the posterior "
        "anchor-minus-Context difference in "
        "$P(Y=1)+P(Y=4)$. A 95\\% credible interval containing "
        "zero was classified as uncertain. Complete directional "
        "anchoring required higher ratings under the high-numeric "
        "anchor and lower ratings under the low-numeric anchor."
    ),
    wide=True,
)

# ------------------------------------------------------------------
# Appendix Table 1: detailed endpoint contrasts
# ------------------------------------------------------------------

appendix_extremity = extremity.copy()
appendix_extremity["Model"] = (
    appendix_extremity["base_model"].map(model_labels)
)
appendix_extremity["Anchor"] = (
    appendix_extremity["anchor_type"].map(anchor_labels)
)
appendix_extremity["Context endpoint probability"] = (
    appendix_extremity["context_probability_median"]
    .map(lambda x: f"{x:.3f}")
)
appendix_extremity["Anchor endpoint probability"] = (
    appendix_extremity["anchor_probability_median"]
    .map(lambda x: f"{x:.3f}")
)
appendix_extremity[r"$\Delta$"] = (
    appendix_extremity["difference_median"]
    .map(lambda x: f"{x:+.3f}")
)
appendix_extremity[r"95\% CrI"] = appendix_extremity.apply(
    lambda row: (
        f"[{row['difference_l95']:.3f}, "
        f"{row['difference_u95']:.3f}]"
    ),
    axis=1,
)
appendix_extremity[r"$P(\Delta>0)$"] = (
    appendix_extremity[
        "posterior_probability_difference_gt_0"
    ].map(format_probability)
)
appendix_extremity["Interpretation"] = (
    appendix_extremity["conclusion"].map(extremity_labels)
)

appendix_extremity["model_order"] = (
    appendix_extremity["base_model"]
    .map({m: i for i, m in enumerate(model_order)})
)
appendix_extremity["anchor_order"] = (
    appendix_extremity["anchor_type"]
    .map({"ANCHOR_EXAMPLE": 1, "ANCHOR_WORD": 2})
)

appendix_extremity = (
    appendix_extremity
    .sort_values(["model_order", "anchor_order"])
    [[
        "Model",
        "Anchor",
        "Context endpoint probability",
        "Anchor endpoint probability",
        r"$\Delta$",
        r"95\% CrI",
        r"$P(\Delta>0)$",
        "Interpretation",
    ]]
)

appendix_extremity_path = (
    output_dir
    / "rq3_feasibility_extremity_appendix.tex"
)

save_regular_table(
    table=appendix_extremity,
    path=appendix_extremity_path,
    caption=(
        "RQ3 feasibility: primary-prior endpoint-probability "
        "contrasts for the word and example anchors."
    ),
    label="tab:rq3_feasibility_extremity_appendix",
    column_format="llcccccc",
    note=(
        "Endpoint probability is $P(Y=1)+P(Y=4)$. "
        "$\\Delta$ is Anchor minus Context. Positive values indicate "
        "greater endpoint concentration under the anchor."
    ),
    wide=True,
)

# ------------------------------------------------------------------
# Appendix Table 2: detailed numeric-anchor patterns
# ------------------------------------------------------------------

numeric_appendix = numeric.copy()
numeric_appendix["Model"] = (
    numeric_appendix["base_model"].map(model_labels)
)
numeric_appendix["High numeric OR [95\\% CrI]"] = (
    numeric_appendix.apply(
        lambda row: format_or_interval(row, "high"),
        axis=1,
    )
)
numeric_appendix["High-arm interpretation"] = (
    numeric_appendix["high_conclusion"]
    .replace({
        "higher under Anchor": "Higher",
        "lower under Anchor": "Lower",
        "uncertain": "Uncertain",
    })
)
numeric_appendix["Low numeric OR [95\\% CrI]"] = (
    numeric_appendix.apply(
        lambda row: format_or_interval(row, "low"),
        axis=1,
    )
)
numeric_appendix["Low-arm interpretation"] = (
    numeric_appendix["low_conclusion"]
    .replace({
        "higher under Anchor": "Higher",
        "lower under Anchor": "Lower",
        "uncertain": "Uncertain",
    })
)
numeric_appendix["Combined pattern"] = (
    numeric_appendix["anchoring_pattern"].map(pattern_labels)
)
numeric_appendix["model_order"] = (
    numeric_appendix["base_model"]
    .map({m: i for i, m in enumerate(model_order)})
)

numeric_appendix = (
    numeric_appendix
    .sort_values("model_order")
    [[
        "Model",
        "High numeric OR [95\\% CrI]",
        "High-arm interpretation",
        "Low numeric OR [95\\% CrI]",
        "Low-arm interpretation",
        "Combined pattern",
    ]]
)

appendix_numeric_path = (
    output_dir
    / "rq3_feasibility_numeric_patterns_appendix.tex"
)

save_regular_table(
    table=numeric_appendix,
    path=appendix_numeric_path,
    caption=(
        "RQ3 feasibility: primary-prior high- and low-numeric "
        "anchor patterns."
    ),
    label="tab:rq3_feasibility_numeric_patterns_appendix",
    column_format="lccccc",
    note=(
        "ORs compare each numeric anchor with Context. Complete "
        "directional anchoring required a supported increase under "
        "the high-numeric anchor and a supported decrease under the "
        "low-numeric anchor."
    ),
    wide=True,
)

# ------------------------------------------------------------------
# Appendix Table 3: robustness summary
# ------------------------------------------------------------------

robustness_table = pd.DataFrame({
    "Assessment": [
        "Endpoint-effect direction",
        "Endpoint-effect classification",
        "Numeric anchoring pattern",
    ],
    "Stable comparisons": [
        int(extremity_robustness["direction_stable"].sum()),
        int(extremity_robustness["conclusion_stable"].sum()),
        int(numeric_robustness["pattern_stable"].sum()),
    ],
    "Total comparisons": [
        len(extremity_robustness),
        len(extremity_robustness),
        len(numeric_robustness),
    ],
})

robustness_table["Stability rate"] = (
    robustness_table["Stable comparisons"]
    / robustness_table["Total comparisons"]
).map(lambda x: f"{100*x:.1f}\\%")

appendix_robustness_path = (
    output_dir
    / "rq3_feasibility_anchor_robustness_appendix.tex"
)

save_regular_table(
    table=robustness_table,
    path=appendix_robustness_path,
    caption=(
        "RQ3 feasibility: robustness of anchor-response patterns "
        "across prior specifications."
    ),
    label="tab:rq3_feasibility_anchor_robustness_appendix",
    column_format="lccc",
    note=(
        "Stability was evaluated across the regularizing, primary, "
        "and weaker population-level coefficient priors."
    ),
)

print("RQ3 feasibility LaTeX tables saved in:")
print(output_dir.resolve())
display(main_table)
display(appendix_extremity)
display(numeric_appendix)
display(robustness_table)

RQ3 feasibility LaTeX tables saved in:
/Users/HP/Documents/second-publication/current-analysis/re-run/rq3-rerun/feas-ctx-anchor-working/bayesian-results/anchor-response-patterns/latex-tables


,Model,Example $\Delta_{\mathrm{endpoint}}$ [95\% CrI],Word $\Delta_{\mathrm{endpoint}}$ [95\% CrI],Numeric anchoring pattern
0,Gemma3:12B,"+0.001 [-0.005, 0.030]; Uncertain","+0.001 [-0.005, 0.031]; Uncertain",No supported adjustment
1,LLaMa-Pro,"-0.083 [-0.161, 0.018]; Uncertain","+0.104 [-0.038, 0.275]; Uncertain",No supported adjustment
2,Mistral,"+0.001 [-0.005, 0.035]; Uncertain","+0.006 [-0.004, 0.066]; Uncertain",No supported adjustment
3,Phi-4,"+0.001 [-0.005, 0.030]; Uncertain","+0.001 [-0.005, 0.031]; Uncertain",No supported adjustment


,Model,Anchor,Context endpoint probability,Anchor endpoint probability,$\Delta$,95\% CrI,$P(\Delta>0)$,Interpretation
0,Gemma3:12B,Example,0.008,0.010,+0.001,"[-0.005, 0.030]",0.688,Uncertain
1,Gemma3:12B,Word,0.008,0.010,+0.001,"[-0.005, 0.031]",0.683,Uncertain
2,LLaMa-Pro,Example,0.269,0.185,-0.083,"[-0.161, 0.018]",0.047,Uncertain
3,LLaMa-Pro,Word,0.269,0.373,+0.104,"[-0.038, 0.275]",0.921,Uncertain
4,Mistral,Example,0.010,0.013,+0.001,"[-0.005, 0.035]",0.701,Uncertain
5,Mistral,Word,0.010,0.017,+0.006,"[-0.004, 0.066]",0.823,Uncertain
6,Phi-4,Example,0.008,0.010,+0.001,"[-0.005, 0.030]",0.679,Uncertain
7,Phi-4,Word,0.008,0.010,+0.001,"[-0.005, 0.031]",0.688,Uncertain


,Model,High numeric OR [95\% CrI],High-arm interpretation,Low numeric OR [95\% CrI],Low-arm interpretation,Combined pattern
0,Gemma3:12B,"1.257 [0.163, 10.513]",Uncertain,"1.260 [0.168, 11.360]",Uncertain,No supported adjustment
1,LLaMa-Pro,"1.509 [0.685, 3.266]",Uncertain,"1.189 [0.549, 2.523]",Uncertain,No supported adjustment
2,Mistral,"1.168 [0.160, 9.160]",Uncertain,"1.164 [0.156, 8.832]",Uncertain,No supported adjustment
3,Phi-4,"1.208 [0.166, 10.510]",Uncertain,"1.228 [0.151, 10.851]",Uncertain,No supported adjustment


,Assessment,Stable comparisons,Total comparisons,Stability rate
0,Endpoint-effect direction,8,8,100.0\%
1,Endpoint-effect classification,8,8,100.0\%
2,Numeric anchoring pattern,4,4,100.0\%
